⚙ ***yolov8_object_detection***

**YOLOv8** is a real-time object detection model that processes the entire image in a single forward pass, making it fast and accurate.
It's the current industry standard, has excellent pretrained weights, and **the Ultralytics library** makes fine-tuning and metric logging straightforward.

In [ ]:
!python -m pip install ultralytics
from ultralytics import YOLO
print("YOLOv8 ready ✅")

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO
from IPython.display import Image as IPImage

# Load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")  # 'n' = nano, smallest and fastest

# Run inference on a sample image from the web
results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    conf=0.5,        # confidence threshold: anything below 50% confidence gets ignored
    save=True,       # saves the output image with bounding boxes
    show=False
)

# Print what it detected
for result in results:
    print("Detected objects:")
    for box in result.boxes:
        class_id = int(box.cls)
        confidence = float(box.conf)
        class_name = model.names[class_id]
        print(f"  - {class_name}: {confidence:.2f}")



IPImage('/content/runs/detect/predict/bus.jpg')

### Dataset
- Source: Roboflow Universe – Helmet Detection Dataset
- 1,540 images, 4 classes: Helmet, NoHelmet, Motorbike, PNumber
- Format: YOLOv8 (pre-split into train/val/test)

### Approach
1. Ran inference with pretrained YOLOv8n (trained on COCO 80 classes)
2. Fine-tuned on the helmet dataset for domain-specific detection
3. Logged and compared metrics before and after fine-tuning

### Metrics Tracked
| Metric | Meaning |
|---|---|
| mAP@50 | Primary detection accuracy (box + class combined) |
| Precision | % of detections that were correct |
| Recall | % of real objects that were found |


In [ ]:
# Step 1: Install and download dataset
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="re0L9uT0rHD3RdeTOQdc")
project = rf.workspace("helmet-detection-nmzik").project("helmet-detection-nsbwm")
version = project.version(2)
dataset = version.download("yolov8")

Now, I fine-tuned YOLOv8n on a 1,500-image helmet safety dataset for 30 epochs, achieving mAP50 of 86.2%, precision of 85.8%, and recall of 81.2%. The model performed best on motorbike detection at 92.4% mAP.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Fine-tune on helmet dataset
results = model.train(
    data="/content/Helmet-detection-2/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    name="helmet_finetune",
    plots=True
)

print("Training complete!")

In [ ]:
from IPython.display import Image as IPImage
IPImage('/content/runs/detect/helmet_finetune/results.png')

⚡ As you notice, The training and validation losses both decreased in parallel, which confirms the model wasn't overfitting. mAP50 plateaued around epoch 15, suggesting 30 epochs was a reasonable choice — more epochs likely wouldn't have helped significantly without more data.


Now run this to test the model on a real image:

In [ ]:
# Download a test image directly
!wget -O test_helmet.jpg "https://images.unsplash.com/photo-1558618666-fcd25c85cd64?w=640"

# Test your fine-tuned model
from ultralytics import YOLO
model = YOLO('/content/runs/detect/helmet_finetune/weights/best.pt')

results = model.predict(
    source="test_helmet.jpg",
    conf=0.4,
    save=True,
    name="helmet_test"
)

for result in results:
    for box in result.boxes:
        print(f"{model.names[int(box.cls)]}: {float(box.conf):.2f}")


from IPython.display import Image as IPImage
import glob
latest = sorted(glob.glob('/content/runs/detect/helmet_test*'))[-1]
IPImage(glob.glob(f'{latest}/*.jpg')[0])

‼❗ Insights
When tested on out-of-distribution images — close-up indoor shots rather than road scenes — the model still correctly classified the person as not wearing a helmet, though with lower confidence (48%) compared to in-distribution test images. This shows the model learned the concept of helmet vs no helmet, not just the road scene context.

In [ ]:
from google.colab import files
files.download('/content/results.png')